##### IMPORTS AND FUNCTION DEFINITIONS

In [1]:
import pandas as pd
from pyannote.metrics.diarization import DiarizationErrorRate
from pyannote.core import Segment, Timeline, Annotation
from itertools import product
import os

# Use below only to analyze single speaker diarization
# TARGET_MODEL_SPEAKER = 'speaker_1'
# TARGET_GOLD_SPEAKER = 'SPEAKER_00'

def convert_model_csv_to_rrtm(model_root_folder, csv_filename):
    """
    Convert the diarization model's CSV output to RTTM format for scoring with pyannote diarization error metrics.
    
    Args:
        model_root_folder (str): The root folder where the model's CSV output is located.
        csv_filename (str): The name of the CSV file to convert.
    Returns:
        None: Writes the RTTM file to the same directory as the CSV file in the diarization model's output folder.
    """

    # Handle P5-S8 to trim upto where gold label exists.
    P5_S8_LIMIT = 1719.725

    with open(os.path.join(model_root_folder, csv_filename[:-4]+".rttm"), 'w') as rttm_file:
        df = pd.read_csv(f"{model_root_folder}/{csv_filename}")
        for _, row in df.iterrows():
            # if(row['speaker'] == TARGET_MODEL_SPEAKER):
                begin = float(row['start_time'])
                end = float(row['end_time'])
                speaker = row['speaker']

                # Special handling for p5-s8
                if "p5-s8" in csv_filename:
                    # Ignore segments that start after gold labels end
                    if begin >= P5_S8_LIMIT:
                         continue
                    # Truncate segments that cross the boundary
                    end = min(end, P5_S8_LIMIT)

                # RTTM format: SPEAKER <file> <channel> <begin> <duration> <conf> <speech_type> <speaker_type> <speaker_name>
                duration = end - begin
                if duration <= 0:
                     continue
                rttm_line = f"SPEAKER {begin:.3f} {duration:.3f} <NA> <NA> <NA> {speaker}\n"
                
                rttm_file.write(rttm_line)

def convert_gold_label_txt_to_rrtm(gold_label_root_folder, txt_filename):
    """
    Convert the gold label diarization annotations from a TXT file (from ELAN) to RTTM format for scoring with pyannote diarization error metrics.
    
    Args:
        gold_label_root_folder (str): The root folder where the gold label TXT file is located.
        txt_filename (str): The name of the TXT file to convert.
    Returns:
        None: Writes the RTTM file to the same directory as the TXT file in the gold label annotations folder.
    """

    df = pd.read_csv(os.path.join(gold_label_root_folder, txt_filename), sep='\t')
    # Create RTTM output
    with open(os.path.join(gold_label_root_folder, txt_filename[:-4]+".rttm"), 'w') as rttm_file:
        for idx, row in df.iterrows():
            # if(row['speaker'] == TARGET_GOLD_SPEAKER):
                begin = float(row['speech_start'])
                end = float(row['speech_end'])
                speaker = row['speaker']
                # RTTM format: SPEAKER <file> <channel> <begin> <duration> <conf> <speech_type> <speaker_type> <speaker_name>
                duration = end - begin
                rttm_line = f"SPEAKER {begin:.3f} {duration:.3f} <NA> <NA> <NA> {speaker}\n"
                rttm_file.write(rttm_line)

def load_rttm_to_annotation(path):
    """Load an RTTM file and convert it to a pyannote.core.Annotation object.
    Args:
        path (str): Path to the RTTM file.
    Returns:
        pyannote.core.Annotation: The loaded annotation."""
    ann = Annotation()
    with open(path, 'r') as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if parts[0] != 'SPEAKER':
                continue
            # handle both custom and standard RTTM token positions
            try:
                start = float(parts[1])
                dur = float(parts[2])
                speaker = parts[-1]
            except Exception:
                start = float(parts[3])
                dur = float(parts[4])
                speaker = parts[-1]
            ann[Segment(start, start + dur)] = speaker
    return ann

def der_component_conversion(der_dict):
    """Convert the DER components from a dictionary to a more human-readable format.
    Args:
        der_dict (dict): The dictionary containing DER components.
    Returns:
        dict: A dictionary with human-readable DER components."""
    return {
        'missed_speech_percent': f"{100*der_dict['missed detection']/der_dict['total']:.2f}",
        'false_alarm_percent': f"{100*der_dict['false alarm']/der_dict['total']:.2f}",
        'confusion_percent': f"{100*der_dict['confusion']/der_dict['total']:.2f}",
        "correct_percent": f"{100*der_dict['correct']/der_dict['total']:.2f}",
        "der": f"{100*der_dict['diarization error rate']:.2f}"
    }

##### SPECIFY FILE_ID, VARIANT_TYPES AND RESULT FORMAT

In [2]:
ids = [
    "p5-s8",
    "p5-s10",
    "p7-s8",
    "p7-s16",
    "p9-s3-1",
    "p9-s9",
    "p11-s4",
    "p11-s11",
    "p12-s3",
    "p12-s6",
    "p17-s2",
    "p17-s6",
    "p18-s15",
    "p18-s17",
]

MODEL_TYPES = [ 
    "ClusteringDiarizer",
    "NeuralDiarizer",
]

VAD_MODELS = [
    "vad_marblenet",
    "vad_multilingual_marblenet",
    "vad_telephony_marblenet",
]

EMBEDDING_MODELS = [
    "titanet_large",
    "ecapa_tdnn",
]

AUDIO_TYPES = [
    "raw_audios",
    "denoised_audios",
]

SPEAKER_MODES = [
    "fixed_num-speakers",
    "auto_num-speakers"
]

OVERLAP_MODES = [
    "overlap",
    "no_overlap",
]

# Cartesian product of all possible combinations
all_configs = list(product(
    MODEL_TYPES,
    AUDIO_TYPES,
    SPEAKER_MODES,
    OVERLAP_MODES,
    VAD_MODELS,
    EMBEDDING_MODELS
))

results = {
    "Model Type": [],
    "Audio Type": [],
    "Speaker Mode": [],
    "Overlap Mode": [],
    "VAD": [],
    "Embedding": [],
    "video_id": [], # Corresponds to the file id in other parts of the script
    "DER": [],
    "False Alarm %": [],
    "Missed Speech %": [],
    "Confusion %": [],
    "Correct %": []
}

##### SPECIFY MAPPING FOR CSV FILES (IF CREATED)

In [3]:
#Mapping for the abbreviations, when creating the CSV file names
MODEL_MAP = {
    "ClusteringDiarizer": "C",
    "NeuralDiarizer": "N"
}

VAD_MAP = {
    "vad_marblenet": "MN",
    "vad_multilingual_marblenet": "MMN",
    "vad_telephony_marblenet": "TMN"
}

EMBEDDING_MAP = {
    "titanet_large": "TL",
    "ecapa_tdnn": "ET"
}

AUDIO_MAP = {
    "raw_audios": "RAW",
    "denoised_audios": "DENOISED"
}

SPEAKER_MAP = {
    "fixed_num-speakers": "FIXED",
    "auto_num-speakers": "AUTO"
}

OVERLAP_MAP = {
    "overlap": "OVERLAP",
    "no_overlap": "NO_OVERLAP"
}


##### GENERATE ALL GOLD LABEL RTTMS AT ONCE

In [4]:
# Path to Reference (Gold Labels) folder
ref_root = "gold_label_diarization_annotations"

# print("Generating gold RTTMs...")

for file_id in ids:

    ref_file_name = f"{file_id}_gold_labels"

    convert_gold_label_txt_to_rrtm(
        ref_root,
        ref_file_name + ".txt"
    )

# print("Done.")

##### LOOP THROUGH COMBINATIONS

In [5]:
os.makedirs("der_results", exist_ok=True)

# Path to Reference (Gold Labels) folder
ref_root = "gold_label_diarization_annotations"

# Loops thorugh all variants
for (
    model_type,
    audio_type,
    speaker_mode,
    overlap_mode,
    vad_model,
    embedding_model
) in all_configs:
    
    # Current configuation's result format to store in CSV output
    config_results = {
        "video_id": [],
        "DER": [],
        "False Alarm %": [],
        "Missed Speech %": [],
        "Confusion %": [],
        "Correct %": []
    }

    # Path to hypothetical (model) diarization outputs
    hyp_root = os.path.join(
        "outputs", 
        model_type,
        audio_type,
        speaker_mode,
        vad_model,
        embedding_model,
        "pred_rttms"
    )

    # Loop through all the audio samples 
    for file_id in ids:
        
        # Path to the sample's Gold Label (Reference) file
        ref_file_name = f"{file_id}_gold_labels"
        
        # Path to the sample's model output (Hypothesis) file
        if audio_type == "denoised_audios":
            hyp_file_name = f"{file_id}_denoised"
        else:
            hyp_file_name = file_id

        try:

            convert_model_csv_to_rrtm(
                hyp_root,
                hyp_file_name + ".csv"
            )

            ref = load_rttm_to_annotation(
                os.path.join(
                    ref_root,
                    ref_file_name + ".rttm"
                )
            )

            hyp = load_rttm_to_annotation(
                os.path.join(
                    hyp_root,
                    hyp_file_name + ".rttm"
                )
            )

            # Decide which overlap mode
            skip_overlap = (overlap_mode == "no_overlap")

            metric = DiarizationErrorRate(
                collar=0.25,
                skip_overlap=skip_overlap
            )

            der = der_component_conversion(
                metric(ref, hyp, detailed=True)
            )

            # Append the current results to the main RESULTS dataframe
            results["video_id"].append(file_id)

            results["Model Type"].append(model_type)
            results["Audio Type"].append(audio_type)
            results["Speaker Mode"].append(speaker_mode)
            results["Overlap Mode"].append(overlap_mode)

            results["VAD"].append(vad_model)
            results["Embedding"].append(embedding_model)

            results["DER"].append(float(der["der"]))
            results["False Alarm %"].append(float(der["false_alarm_percent"]))
            results["Missed Speech %"].append(float(der["missed_speech_percent"]))
            results["Confusion %"].append(float(der["confusion_percent"]))
            results["Correct %"].append(float(der["correct_percent"]))


            # Append the current results to a temporary CONFIG_RESULTS dataframe to store into a seperate CSV file
            config_results["video_id"].append(file_id)
            config_results["DER"].append(float(der["der"]))
            config_results["False Alarm %"].append(float(der["false_alarm_percent"]))
            config_results["Missed Speech %"].append(float(der["missed_speech_percent"]))
            config_results["Confusion %"].append(float(der["confusion_percent"]))
            config_results["Correct %"].append(float(der["correct_percent"]))

        except Exception as e:

            print(
                f"Failed: {model_type},{audio_type},{speaker_mode},{overlap_mode},{vad_model},{embedding_model} | {file_id}"
            )

            print(e)

    config_df = pd.DataFrame(config_results)
    
    # Configure the name of the csv file for current configuration/ model combination 
    csv_name = (
        f"model_{MODEL_MAP[model_type]}"
        f"-vad_{VAD_MAP[vad_model]}"
        f"-embedding_{EMBEDDING_MAP[embedding_model]}"
        f"-audio_{AUDIO_MAP[audio_type]}"
        f"-speaker_{SPEAKER_MAP[speaker_mode]}"
        f"-overlap_{OVERLAP_MAP[overlap_mode]}"
        f"-params_OLD.csv"
    )
    
    # Craete a csv
    config_df.to_csv(
        os.path.join("der_results", csv_name),
        index=False
    )

    print(f"Saved: {csv_name}")


/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 

Saved: model_C-vad_MN-embedding_TL-audio_RAW-speaker_FIXED-overlap_OVERLAP-params_OLD.csv


/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 

Saved: model_C-vad_MN-embedding_ET-audio_RAW-speaker_FIXED-overlap_OVERLAP-params_OLD.csv


/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 

...

/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 

Saved: model_N-vad_TMN-embedding_TL-audio_DENOISED-speaker_AUTO-overlap_NO_OVERLAP-params_OLD.csv


/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(
/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 

Saved: model_N-vad_TMN-embedding_ET-audio_DENOISED-speaker_AUTO-overlap_NO_OVERLAP-params_OLD.csv


/home/interactionlab/anaconda3/envs/der_env/lib/python3.10/site-packages/pyannote/metrics/utils.py:200: UserWarning: 'uem' was approximated by the union of 'reference' and 'hypothesis' extents.
  warnings.warn(


##### PRINT OUTPUTS

In [6]:
# Print the full results table
df = pd.DataFrame(results)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df

,Model Type,Audio Type,Speaker Mode,Overlap Mode,VAD,Embedding,video_id,DER,False Alarm %,Missed Speech %,Confusion %,Correct %
0,ClusteringDiarizer,raw_audios,fixed_num-speakers,overlap,vad_marblenet,titanet_large,p5-s8,49.08,2.95,41.81,4.32,53.87
1,ClusteringDiarizer,raw_audios,fixed_num-speakers,overlap,vad_marblenet,titanet_large,p5-s10,49.20,2.10,37.87,9.23,52.90
2,ClusteringDiarizer,raw_audios,fixed_num-speakers,overlap,vad_marblenet,titanet_large,p7-s8,71.35,2.10,54.50,14.75,30.75
3,ClusteringDiarizer,raw_audios,fixed_num-speakers,overlap,vad_marblenet,titanet_large,p7-s16,47.26,1.47,40.94,4.85,54.21
4,ClusteringDiarizer,raw_audios,fixed_num-speakers,overlap,vad_marblenet,titanet_large,p9-s3-1,85.74,3.20,76.83,5.71,17.46
5,ClusteringDiarizer,raw_audios,fixed_num-speakers,overlap,vad_marblenet,titanet_large,p9-s9,51.78,4.60,44.85,2.33,52.81
6,ClusteringDiarizer,raw_audios,fixed_num-speakers,overlap,vad_marblenet,titanet_large,p11-s4,51.59,3.63,38.71,9.25,52.04
7,ClusteringDiarizer,raw_audios,fixed_num-speakers,overlap,vad_marblenet,titanet_large,p11-s11,42.16,1.54,30.77,9.86,59.37
8,ClusteringDiarizer,raw_audios,fixed_num-speakers,overlap,vad_marblenet,titanet_large,p12-s3,90.49,0.48,86.38,3.64,9.98
9,ClusteringDiarizer,raw_audios,fixed_num-speakers,overlap,vad_marblenet,titanet_large,p12-s6,93.06,0.15,91.16,1.75,7.09


In [7]:
# df[[
#     "Model Type",
#     "Audio Type",
#     "Speaker Mode",
#     "Overlap Mode",
#     "VAD",
#     "Embedding",
#     "video_id", # Corresponds to the file id in other parts of the script
#     "DER",
#     "False Alarm %",
#     "Missed Speech %",
#     "Confusion %",
#     "Correct %"
# ]]

##### GENERATE SUMMARY OF OUTPUT

In [10]:
summary = (
     df.groupby([
        "Model Type",
        "Audio Type",
        "Speaker Mode",
        "VAD",
        "Embedding",
        "Overlap Mode"
    ])
    .agg({
        "DER": ["mean", "median", "std"],
        "False Alarm %": ["mean", "median", "std"],
        "Missed Speech %": ["mean", "median", "std"],
        "Confusion %": ["mean", "median", "std"],
        "Correct %": ["mean", "median", "std"]
    })
)

# summary = summary.sort_values(
#     ("DER", "median")
# )

summary

DER  \
                                                                                                                  mean   
Model Type         Audio Type      Speaker Mode       VAD                        Embedding     Overlap Mode              
ClusteringDiarizer denoised_audios auto_num-speakers  vad_marblenet              ecapa_tdnn    no_overlap    40.767857   
                                                                                               overlap       43.327857   
                                                                                 titanet_large no_overlap    42.722857   
                                                                                               overlap       44.972143   
                                                      vad_multilingual_marblenet ecapa_tdnn    no_overlap    32.743571   
                                                                                               overlap       35.435000   
                                                                                 titanet_large no_overlap    37.412857   
                                                                                               overlap       39.393571   
                                                      vad_telephony_marblenet    ecapa_tdnn    no_overlap    41.696429   
                                                                                               overlap       43.037143   
                                                                                 titanet_large no_overlap    42.990000   
                                                                                               overlap       44.040714   
                                   fixed_num-speakers vad_marblenet              ecapa_tdnn    no_overlap    45.764286   
                                                                                               overlap       47.862143   
                                                                                 titanet_large no_overlap    40.837857   
                                                                                               overlap       43.419286   
                                                      vad_multilingual_marblenet ecapa_tdnn    no_overlap    39.839286   
                                                                                               overlap       41.972857   
                                                                                 titanet_large no_overlap    36.183571   
                                                                                               overlap       38.545714   
                                                      vad_telephony_marblenet    ecapa_tdnn    no_overlap    43.170000   
                                                                                               overlap       44.689286   
                                                                                 titanet_large no_overlap    38.409286   
                                                                                               overlap       40.237143   
                   raw_audios      auto_num-speakers  vad_marblenet              ecapa_tdnn    no_overlap    54.904286   
                                                                                               overlap       57.400714   
                                                                                 titanet_large no_overlap    53.055000   
                                                                                               overlap       55.527857   
                                                      vad_multilingual_marblenet ecapa_tdnn    no_overlap    61.770000   
                                                                                               overlap       64.083571   
                                                                                 titanet_